<a href="https://colab.research.google.com/github/kny1209/test2/blob/main/AI/DR-08079_%EA%B5%AC%EB%82%98%EC%98%81_AI%EA%B0%9C%EB%A1%A0_17%EC%B0%A8%EC%8B%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q "datasets>=3.0.1" "transformers>=4.45.2"

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

문제 1 (단답형 주관식)

자연어 처리(NLP)에서는 문장마다 단어의 개수(길이)가 다릅니다. LSTM 같은 모델에 미니 배치(Mini-Batch)로 데이터를 입력하기 위해, 모든 문장의 길이를 동일하게 맞춰주는 "패딩(Padding)" 작업이 필요합니다.

이 패딩(Padding) 작업이 왜 필요하며, 어떤 방식으로 동작하는지 1~2줄로 간략히 서술하시오.

딥러닝 모델(PyTorch 텐서)은 동일한 크기의 행렬 연산을 수행해야 하므로, 하나의 미니 배치 내에 있는 모든 문장(시퀀스)의 길이를 동일한 길이로 통일해서 텐서 형태로 변환 한 후 한번에 처리하기 위해서 필요하다.

배치 내 가장 긴 문장을 기준으로, 그보다 짧은 문장들의 뒷부분(또는 앞부분)에 [PAD]라는 특수 토큰(보통 인덱스 0)을 채워 넣어 길이를 맞춘다.

문제 2 (단답형 주관식)

NLP 모델의 첫 번째 계층으로 사용되는 nn.Embedding의 역할은 무엇인가요? '토큰 인덱스(Token Index)'와 '벡터(Vector)'라는 키워드를 사용하여 서술하시오.

토큰 인덱스(ex. 3,5,2)로 구성된 시퀀스를 입력 받아, 각 인덱스에 해당하는 고차원 밀집 벡터로 변환하는 역할을 한다.

이 벡터는 단어의 의미적 특징을 학습한다.

토큰 인덱스를 동일한 크기의 벡터로 변환하여 의미적으로 유사한 단어들을 가까이 위치한다. (ex. love, like)

문제 3 (실습 문제 - 코드 빈칸 채우기)

nn.Module을 상속받아 기본적인 LSTM 분류 모델(BasicLSTM)을 정의하는 코드입니다. (19차시 BiLSTM 모델의 단순화 버전)

[요구사항]
- __init__: vocab_size, embed_size, hidden_size를 인자로 받아 3개의 계층을 정의합니다.
  - nn.Embedding
  - nn.LSTM (단방향, batch_first=True)
  - nn.Linear (LSTM의 hidden_size를 받아 num_classes로 출력)

- forward: embedding -> lstm -> linear 순서로 통과시키되, lstm의 마지막 시점(last time step)의 은닉 상태(hidden state)를 linear 계층에 입력합니다.


4개의 빈칸 (# TODO: ...)을 채우시오.

In [ ]:
import torch
import torch.nn as nn

class BasicLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_classes):
        super(BasicLSTM, self).__init__()

        # TODO: 1. 단어 사전을 입력받아 임베딩 벡터를 출력하는 계층
        self.embedding = nn.Embedding(vocab_size, embed_size)

        # TODO: 2. 임베딩 벡터를 입력받아 은닉 상태를 출력하는 LSTM 계층
        self.lstm = nn.LSTM(
            input_size = embed_size,
            hidden_size = hidden_size,
            batch_first = True,  # 입력 텐서 형식을 (배치 크기, 시퀀스 길이, ...)로 설정, 입력 텐서의 첫 번째 차원이 배치 크기임을 명시 (중요)
            bidirectional = False # 단방향
        )

        # TODO: 3. LSTM의 은닉 상태를 입력받아 클래스 점수(logits)를 출력하는 선형 계층
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x shape: (배치 크기, 시퀀스 길이)

        emb = self.embedding(x)
        # emb shape: (배치 크기, 시퀀스 길이, embed_size)

        # TODO: 4. LSTM 실행. (hn은 마지막 은닉 상태)
        # lstm_out shape: (배치 크기, 시퀀스 길이, hidden_size)
        # hn shape: (1, 배치 크기, hidden_size)
        lstm_out, (hn, cn) = self.lstm(emb)

        # 마지막 은닉 상태(hn)를 fc 계층에 입력
        # hn[0] shape: (배치 크기, hidden_size)
        logits = self.fc(hn[0])     # 마지막 시점의 은닉 상태
        # lstm (전체 시퀀스 출력, (마지막 은닉 상태 hn), (마지막 셀 상태 cn))
        # shape:[배치 크기, 은닉 크기] >> fc계층 전달

        return logits

# --- 테스트 코드 (수정 불필요) ---
VOCAB_SIZE = 1000
EMBED_SIZE = 32
HIDDEN_SIZE = 64
NUM_CLASSES = 2

model = BasicLSTM(VOCAB_SIZE, EMBED_SIZE, HIDDEN_SIZE, NUM_CLASSES)
print(model)

dummy_input = torch.randint(0, VOCAB_SIZE, (4, 10)) # (배치 4, 시퀀스 길이 10)
output = model(dummy_input)
print(f"\n입력 Shape: {dummy_input.shape}")
print(f"출력 Shape: {output.shape}") # (배치 4, 클래스 2)

BasicLSTM(
  (embedding): Embedding(1000, 32)
  (lstm): LSTM(32, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=2, bias=True)
)

입력 Shape: torch.Size([4, 10])
출력 Shape: torch.Size([4, 2])


문제 4 (실습 문제 - 코드 빈칸 채우기)

transformers 라이브러리의 AutoTokenizer를 사용하여 텍스트를 토큰화하고, DataCollatorWithPadding을 사용하여 동적 패딩(dynamic padding)을 준비하는 코드입니다.

3개의 빈칸 (# TODO: ...)을 채우시오.

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

MODEL_NAME = 'distilbert-base-uncased' # 사용할 모델/토크나이저 이름

# TODO: 1. 'distilbert-base-uncased' 모델의 'tokenizer' 로드
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# --- 가상 데이터 (수정 불필요) ---
sentences = ["This is the first sentence.", "This one is shorter."]
# ----------------------------

# TODO: 2. 'tokenizer'를 사용하여 'sentences'를 토큰화 (인덱스 변환)
# (패딩은 적용하지 않음)
# tokenized_inputs = tokenizer(sentences) # 원본
tokenized_inputs = [tokenizer(s) for s in sentences] # 수정: 각 문장을 개별 토큰화하여 리스트로 만듦

# TODO: 3. 'tokenizer'를 기반으로 동적 패딩을 수행하는 'data_collator' 생성
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


# --- 결과 확인 (수정 불필요) ---
print("--- 토크나이저 적용 결과 (패딩 전) ---")
# print(tokenized_inputs['input_ids']) # 원본
print([t['input_ids'] for t in tokenized_inputs]) # 수정: 리스트의 각 BatchEncoding에서 input_ids 추출

# (DataCollator가 배치 데이터를 받아 패딩을 적용하는 예시)
batch = data_collator([tokenized_inputs[0], tokenized_inputs[1]])
print("\n--- DataCollator 적용 결과 (패딩 후) ---")
print(batch['input_ids'])
print(f"Padding Token ID: {tokenizer.pad_token_id}")

--- 토크나이저 적용 결과 (패딩 전) ---
[[101, 2023, 2003, 1996, 2034, 6251, 1012, 102], [101, 2023, 2028, 2003, 7820, 1012, 102]]

--- DataCollator 적용 결과 (패딩 후) ---
tensor([[ 101, 2023, 2003, 1996, 2034, 6251, 1012,  102],
        [ 101, 2023, 2028, 2003, 7820, 1012,  102,    0]])
Padding Token ID: 0


문제 5 (실습 문제 - 코드 빈칸 채우기)

양방향 LSTM은 순방향(forward)과 역방향(backward)의 마지막 은닉 상태 2개를 반환합니다. 이 두 은닉 상태를 연결(concatenate)하여 분류기(fc)에 전달해야 합니다.

hn 텐서(Shape: [2, 배치 크기, 은닉 크기])가 주어졌을 때, 3개의 빈칸 (# TODO: ...)을 채워 두 은닉 상태를 연결하시오.

In [ ]:
import torch
import torch.nn as nn

# --- 가상 BiLSTM 모델 (수정 불필요) ---
class BiLSTM_Forward(nn.Module):
    def __init__(self, hidden_size, num_classes):
        super().__init__()
        # (Embedding, LSTM 계층이 있다고 가정)
        # self.lstm = nn.LSTM(..., bidirectional=True)

        # Bi-LSTM이므로 hidden_size * 2를 입력받음
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, hn):
        # hn: LSTM의 마지막 은닉 상태 (Shape: [2, 4, 64])
        # (num_directions * num_layers, batch_size, hidden_size)
        # (여기서는 num_layers=1, num_directions=2 라고 가정)

        # TODO: 1. 순방향(forward) 마지막 은닉 상태 (hn[0]) 추출
        hn_fwd = hn[0]  # hn[-2]

        # TODO: 2. 역방향(backward) 마지막 은닉 상태 (hn[1]) 추출
        hn_bwd = hn[1] # hn[-1]

        # TODO: 3. 'torch.cat'을 사용하여 두 은닉 상태를 1번 차원(dim=1) 기준으로 연결
        # (Shape: [4, 64]와 [4, 64] -> [4, 128])    128차원 벡터가 문장의 순/역 방향 정보를 모두 담은 최종 특징 벡터 >> fc계층 입력
        hidden = torch.cat((hn_fwd, hn_bwd), dim=1)

        logits = self.fc(hidden)
        return logits, hidden
# -----------------------------------------------

# --- 테스트 코드 (수정 불필요) ---
HIDDEN_SIZE = 64
model = BiLSTM_Forward(HIDDEN_SIZE, 2)
# 가상 hn 텐서 (방향 2, 배치 4, 은닉 64)
dummy_hn = torch.randn(2, 4, HIDDEN_SIZE)

logits, concatenated_hidden = model(dummy_hn)
print(f"입력 hn Shape: {dummy_hn.shape}")
print(f"연결된 Hidden Shape: {concatenated_hidden.shape}")
print(f"최종 Logits Shape: {logits.shape}")

입력 hn Shape: torch.Size([2, 4, 64])
연결된 Hidden Shape: torch.Size([4, 128])
최종 Logits Shape: torch.Size([4, 2])


In [ ]:
# EOS